<a href="https://colab.research.google.com/github/Varada-Kachroo/HimalayanGuardian/blob/main/Part3_Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3 — GLOF Risk Prediction ML
Builds on the Part 1 outputs in Google Drive (`HimalayanGuardian/raw` and `/processed`).
Nothing from Part 1 or Part 2 is modified. This notebook only READS those files and WRITES new ones
into `processed/` and `processed/export/`.

Steps: setup -> load -> feature table -> train (RF / XGBoost / AutoML baseline) with MLflow ->
test on South Lhonak -> score Tista lakes -> forecasting -> Isolation Forest -> export for Part 4.

## 0. Setup (run once per Colab session)

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=180000)
    get_ipython().system('pip install -q xgboost "flaml[automl]" mlflow statsmodels scikit-learn geopandas shapely joblib')

BASE = '/content/drive/MyDrive/HimalayanGuardian' if IN_COLAB else os.environ.get('HG_BASE', './HimalayanGuardian')
RAW, PROC = f'{BASE}/raw', f'{BASE}/processed'
EXPORT, MODELS = f'{PROC}/export', f'{PROC}/models'
for d in (EXPORT, MODELS):
    os.makedirs(d, exist_ok=True)

# ---- switches ----
USE_EE = IN_COLAB           # sample slope from SRTM via Earth Engine (needs your EE login). Set False to skip.
EE_PROJECT = 'southern-frame-509805-f2'
NEG_RATIO = 3               # negatives sampled per positive for the region-matched training set
RANDOM_STATE = 42
METRIC_CRS = 'ESRI:102028'  # Asia South Albers equal-area (metres) - fine for the whole Himalaya
print('BASE =', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE = /content/drive/MyDrive/HimalayanGuardian


In [ ]:
import requests

url = "https://stacks.stanford.edu/object/nf183jx2059"
response = requests.get(url)

with open('/content/drive/MyDrive/HimalayanGuardian/raw/rgi_southeast_asia.zip', 'wb') as f:
    f.write(response.content)

print("Downloaded:", len(response.content), "bytes")

Downloaded: 15576391 bytes


In [ ]:
import zipfile, os

RAW = '/content/drive/MyDrive/HimalayanGuardian/raw'

# Unzip ICIMOD glacial lakes (if not already extracted in Drive)
lake_zip = [f for f in os.listdir(RAW) if 'Glacial Lakes' in f or 'glacial_lake' in f.lower()]
if lake_zip and not os.path.exists(f'{RAW}/glacial_lake_hkh_2005'):
    with zipfile.ZipFile(f'{RAW}/{lake_zip[0]}') as z:
        z.extractall(f'{RAW}/glacial_lake_hkh_2005')
    print("Extracted glacial lakes to Drive")

# Unzip RGI glaciers (if not already extracted in Drive)
rgi_zip = [f for f in os.listdir(RAW) if 'rgi' in f.lower() and f.endswith('.zip')]
if rgi_zip and not os.path.exists(f'{RAW}/rgi_data'):
    with zipfile.ZipFile(f'{RAW}/{rgi_zip[0]}') as z:
        z.extractall(f'{RAW}/rgi_data')
    print("Extracted RGI glaciers to Drive")

for root, dirs, files in os.walk(RAW):
    for f in files:
        if f.endswith('.shp'):
            print(os.path.join(root, f))

Extracted RGI glaciers to Drive
/content/drive/MyDrive/HimalayanGuardian/raw/glacial_lake_hkh_2005/glacial_lake_hkh_2005/data/GlacialLake_5basins_HKH.shp
/content/drive/MyDrive/HimalayanGuardian/raw/rgi_data/15_rgi50_SouthAsiaEast.shp


In [ ]:
import os
RAW = '/content/drive/MyDrive/HimalayanGuardian/raw'
for f in os.listdir(RAW):
    print(f)

osm_villages.json
osm_valley.json
The Status of Glacial Lakes in the Hindu Kush Himalaya.zip
flood_path_rivers.json
osm_rivers.json
glacial_lake_hkh_2005


## 1. Load Part 1 outputs

In [ ]:
import glob, json, warnings
import numpy as np, pandas as pd, geopandas as gpd
warnings.filterwarnings('ignore')

def find_file(name, roots):
    for r in roots:
        hits = glob.glob(os.path.join(r, '**', name), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f'{name} not found under {roots}. Re-run the matching Part 1 cell or copy the file there.')

roots = ['/content', RAW] if IN_COLAB else [RAW]
lakes_shp = find_file('GlacialLake_5basins_HKH.shp', roots)
glac_shp = find_file('15_rgi50_SouthAsiaEast.shp', roots)

burst = pd.read_csv(f'{PROC}/himalaya_burst_lakes.csv')            # Part 1 positives (162 lakes, incl. South Lhonak)
neg_tista = pd.read_csv(f'{PROC}/negative_lakes.csv')              # Part 1 negatives (Tista only)
villages = pd.read_csv(f'{PROC}/flood_path_villages.csv')          # Part 1 flood-path villages

inv = gpd.read_file(lakes_shp)
inv = inv.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
inv['lake_id'] = ['HKH-%05d' % i for i in range(len(inv))]
# lake points from the lat/lon columns (avoids polygon CRS issues)
inv_pts = gpd.GeoDataFrame(inv.drop(columns='geometry'),
                           geometry=gpd.points_from_xy(inv['Longitude'], inv['Latitude']), crs=4326)
# ICIMOD 'Area' is usually m^2. Convert to km^2 if the values look like m^2.
AREA_SCALE = 1e-6 if inv_pts['Area'].median() > 1000 else 1.0
inv_pts['area_km2'] = inv_pts['Area'] * AREA_SCALE
inv_pts['type_clean'] = (inv_pts['Type'].astype(str).str.strip().str.lower()
                         .str.replace(r'[^a-z0-9]+', '_', regex=True))

glac = gpd.read_file(glac_shp).to_crs(METRIC_CRS)
glac['g_area_km2'] = glac.geometry.area / 1e6
glac = glac[['g_area_km2', 'geometry']]

print('Inventory lakes:', len(inv_pts), '| area scale:', AREA_SCALE, '| glaciers:', len(glac))
print('Burst lakes:', len(burst), '| Tista negatives (Part 1):', len(neg_tista))
print(inv_pts['Basin'].value_counts().head(8))

Inventory lakes: 25614 | area scale: 1.0 | glaciers: 13119
Burst lakes: 162 | Tista negatives (Part 1): 359
Basin
Yarlung Zangbo    6880
Upper Indus       3257
Dihang            2826
Koshi             2087
Manas             1898
Kabul             1380
Karnali           1204
Panjnad           1052
Name: count, dtype: int64


## 2. Attach lake attributes to the burst lakes, and pick region-matched negatives

**Why:** Part 1 flagged that positives come from the whole Himalaya while negatives came only from Tista.
A model trained like that can learn "Tista = safe". Here negatives are sampled from the same region the
positives come from, and latitude/longitude are never used as model features.
The Part 1 Tista-only negatives are kept only for an ablation check later.

In [ ]:
inv_m = inv_pts.to_crs(METRIC_CRS)

is_lhonak = burst['lat'].between(27.90, 27.93) & burst['lon'].between(88.18, 88.20)   # same rule as Part 1
burst['is_test'] = is_lhonak.values
burst_pts = gpd.GeoDataFrame(burst, geometry=gpd.points_from_xy(burst['lon'], burst['lat']), crs=4326).to_crs(METRIC_CRS)

# nearest inventory lake within 3 km of each burst coordinate
m = gpd.sjoin_nearest(burst_pts[['lake_key', 'is_test', 'geometry']],
                      inv_m[['lake_id', 'geometry']], how='left', max_distance=3000, distance_col='match_m')
m = m.drop_duplicates('lake_key')
burst_matched = m.dropna(subset=['lake_id']).drop(columns=['index_right', 'geometry'])
print(f'Burst lakes matched to inventory: {len(burst_matched)} / {len(burst)} '
      f'(unmatched ones are dropped - no area/altitude available for them)')

# distance of every inventory lake to the nearest known burst coordinate
d = gpd.sjoin_nearest(inv_m[['lake_id', 'geometry']], burst_pts[['geometry']], how='left', distance_col='dist_burst_m')
d = d.drop_duplicates('lake_id').set_index('lake_id')['dist_burst_m']
inv_pts['dist_burst_m'] = inv_pts['lake_id'].map(d)
inv_pts['near_burst'] = inv_pts['dist_burst_m'] < 2000

pos_ids = set(burst_matched.loc[~burst_matched['is_test'], 'lake_id'])
test_ids = set(burst_matched.loc[burst_matched['is_test'], 'lake_id'])

# candidate negatives: same bounding box as the positives (+0.5 deg), not near any burst, with valid attributes
lat_lo, lat_hi = burst['lat'].min() - 0.5, burst['lat'].max() + 0.5
lon_lo, lon_hi = burst['lon'].min() - 0.5, burst['lon'].max() + 0.5
cand = inv_pts[(inv_pts['Latitude'].between(lat_lo, lat_hi)) & (inv_pts['Longitude'].between(lon_lo, lon_hi))
               & (~inv_pts['near_burst']) & inv_pts['area_km2'].notna() & inv_pts['Altitude'].notna()
               & (~inv_pts['lake_id'].isin(pos_ids | test_ids))]
n_neg = min(len(cand), NEG_RATIO * len(pos_ids))
neg_sample = cand.sample(n=n_neg, random_state=RANDOM_STATE)
neg_ids = set(neg_sample['lake_id'])
print('Positives (train):', len(pos_ids), '| negatives (region-matched):', len(neg_ids), '| test lake ids:', test_ids)

# Part 1 Tista negatives -> inventory ids (for the ablation only)
nt = gpd.GeoDataFrame(neg_tista, geometry=gpd.points_from_xy(neg_tista['Longitude'], neg_tista['Latitude']), crs=4326).to_crs(METRIC_CRS)
nt = gpd.sjoin_nearest(nt[['geometry']], inv_m[['lake_id', 'geometry']], how='left', max_distance=500)
tista_neg_ids = set(nt['lake_id'].dropna())

Burst lakes matched to inventory: 145 / 162 (unmatched ones are dropped - no area/altitude available for them)
Positives (train): 142 | negatives (region-matched): 426 | test lake ids: {'HKH-08027'}


## 3. Feature table
Features per lake: area, altitude, lake type, distance to nearest glacier, glacier area (nearest + within 5 km),
and (optional) slope from SRTM via Earth Engine. Lake growth from Part 2 is handled separately in section 6.

In [ ]:
tista_all = inv_pts[inv_pts['Basin'] == 'Tista']
need_ids = pos_ids | test_ids | neg_ids | tista_neg_ids | set(tista_all['lake_id'])
sub = inv_pts[inv_pts['lake_id'].isin(need_ids)].copy().reset_index(drop=True)
sub_m = sub.to_crs(METRIC_CRS)

feat = pd.DataFrame({'lake_id': sub['lake_id'],
                     'log_area_km2': np.log1p(sub['area_km2'].astype(float)),
                     'altitude': sub['Altitude'].astype(float)})

# nearest glacier
j = gpd.sjoin_nearest(sub_m[['lake_id', 'geometry']], glac, how='left', distance_col='dist_m').drop_duplicates('lake_id').set_index('lake_id')
feat['dist_glacier_km'] = feat['lake_id'].map(j['dist_m'] / 1000)
feat['glacier_area_km2'] = feat['lake_id'].map(j['g_area_km2'])
# glacier area within 5 km
buf = sub_m[['lake_id', 'geometry']].copy()
buf['geometry'] = buf.geometry.buffer(5000)
jj = gpd.sjoin(buf, glac, how='left', predicate='intersects')
feat['glacier_area_5km_km2'] = feat['lake_id'].map(jj.groupby('lake_id')['g_area_km2'].sum()).fillna(0)

# lake type one-hot
dummies = pd.get_dummies(sub['type_clean'], prefix='type').astype(int)
feat = pd.concat([feat, dummies], axis=1)

# optional: slope via Earth Engine (cached)
def ee_slope(df_pts, cache_path):
    import ee
    try:
        ee.Initialize(project=EE_PROJECT)
    except Exception:
        ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
    slope = ee.Terrain.slope(ee.Image('USGS/SRTMGL1_003'))
    red = ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True)
    rows = []
    recs = df_pts[['lake_id', 'Longitude', 'Latitude']].values.tolist()
    for i in range(0, len(recs), 400):
        chunk = recs[i:i + 400]
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la]).buffer(300), {'lake_id': lid}) for lid, lo, la in chunk])
        out = slope.reduceRegions(fc, red, 30).getInfo()['features']
        rows += [{'lake_id': f['properties']['lake_id'], 'slope_mean': f['properties'].get('mean'),
                  'slope_max': f['properties'].get('max')} for f in out]
    r = pd.DataFrame(rows); r.to_csv(cache_path, index=False)
    return r

slope_cache = f'{PROC}/terrain_slope_cache.csv'
if os.path.exists(slope_cache):
    feat = feat.merge(pd.read_csv(slope_cache), on='lake_id', how='left')
elif USE_EE:
    try:
        feat = feat.merge(ee_slope(sub, slope_cache), on='lake_id', how='left')
    except Exception as e:
        print('Slope step skipped (Earth Engine failed):', e)
else:
    print('Slope step skipped (USE_EE = False and no cache).')

FEATURES = [c for c in feat.columns if c != 'lake_id']
print('Features used:', FEATURES)
table = sub[['lake_id', 'Latitude', 'Longitude', 'Basin', 'Sub_basin', 'area_km2', 'Altitude', 'Type', 'near_burst']].merge(feat, on='lake_id')
table['label'] = np.where(table['lake_id'].isin(pos_ids), 1, np.where(table['lake_id'].isin(neg_ids | tista_neg_ids), 0, -1))
table.to_csv(f'{PROC}/feature_table.csv', index=False)
print(table['label'].value_counts())

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Features used: ['log_area_km2', 'altitude', 'dist_glacier_km', 'glacier_area_km2', 'glacier_area_5km_km2', 'type_e_c_', 'type_e_o_', 'type_i_s_', 'type_m_e_', 'type_m_l_', 'type_m_o_', 'type_o', 'slope_mean', 'slope_max']
label
 0    775
 1    142
-1     34
Name: count, dtype: int64


## 4. Train: Random Forest, XGBoost, AutoML baseline — tracked in MLflow
Spatial cross-validation: lakes are grouped into 0.5-degree cells so near neighbours never sit in both train and test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score
import joblib

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False
    print('xgboost not installed - using sklearn HistGradientBoosting instead')

try:
    import mlflow
    mlflow.set_tracking_uri(f'sqlite:///{BASE}/mlflow.db')
    mlflow.set_experiment('glof_risk')
    HAVE_MLFLOW = True
except Exception as e:
    HAVE_MLFLOW = False
    print('MLflow unavailable:', e)

train = table[table['label'].isin([0, 1]) & table['lake_id'].isin(pos_ids | neg_ids)].reset_index(drop=True)
X, y = train[FEATURES], train['label'].values
groups = ((train['Latitude'] // 0.5).astype(int).astype(str) + '_' + (train['Longitude'] // 0.5).astype(int).astype(str)).values
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pos_w = (y == 0).sum() / max((y == 1).sum(), 1)

def make_models():
    rf = make_pipeline(SimpleImputer(strategy='median'),
                       RandomForestClassifier(n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample',
                                              n_jobs=-1, random_state=RANDOM_STATE))
    if HAVE_XGB:
        gb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                           scale_pos_weight=pos_w, eval_metric='logloss', random_state=RANDOM_STATE)
    else:
        gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, class_weight='balanced',
                                            random_state=RANDOM_STATE)
    return {'random_forest': rf, 'xgboost' if HAVE_XGB else 'hist_gradient_boosting': gb}

def metrics(y_true, p):
    pred = (p >= 0.5).astype(int)
    return {'cv_roc_auc': roc_auc_score(y_true, p), 'cv_pr_auc': average_precision_score(y_true, p),
            'cv_recall': recall_score(y_true, pred), 'cv_precision': precision_score(y_true, pred, zero_division=0)}

results, oof = {}, {}
for name, mdl in make_models().items():
    p = cross_val_predict(mdl, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
    results[name] = metrics(y, p); oof[name] = p
    if HAVE_MLFLOW:
        with mlflow.start_run(run_name=name):
            mlflow.log_params({'model': name, 'n_features': len(FEATURES), 'n_pos': int((y == 1).sum()),
                               'n_neg': int((y == 0).sum()), 'cv': 'StratifiedGroupKFold(5), 0.5deg groups'})
            mlflow.log_metrics(results[name])
    print(name, {k: round(v, 3) for k, v in results[name].items()})

# AutoML baseline (FLAML)
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X, y, task='classification', metric='roc_auc', time_budget=60, verbose=0,
               estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree'], eval_method='cv', split_type='group',
               groups=groups, n_splits=5, seed=RANDOM_STATE)
    results['automl_flaml'] = {'cv_roc_auc': 1 - automl.best_loss}
    print('AutoML baseline best:', automl.best_estimator, '| CV ROC-AUC', round(1 - automl.best_loss, 3))
    if HAVE_MLFLOW:
        with mlflow.start_run(run_name='automl_flaml_baseline'):
            mlflow.log_params({'best_estimator': automl.best_estimator})
            mlflow.log_metric('cv_roc_auc', 1 - automl.best_loss)
except Exception as e:
    print('AutoML baseline skipped:', e)

# Ablation: the Part 1 setup (Tista-only negatives) - expect a suspiciously high score; shown for transparency only
try:
    ab = table[table['lake_id'].isin(pos_ids | tista_neg_ids)].reset_index(drop=True)
    ab_y = np.where(ab['lake_id'].isin(pos_ids), 1, 0)
    ab_g = ((ab['Latitude'] // 0.5).astype(int).astype(str) + '_' + (ab['Longitude'] // 0.5).astype(int).astype(str)).values
    ab_model = list(make_models().values())[1]
    ab_p = cross_val_predict(ab_model, ab[FEATURES], ab_y, cv=cv, groups=ab_g, method='predict_proba')[:, 1]
    print('Ablation (Tista-only negatives) CV ROC-AUC:', round(roc_auc_score(ab_y, ab_p), 3),
          '- compare with region-matched above; a big gap = geographic shortcut')
except Exception as e:
    print('Ablation skipped:', e)

2026/10/09 07:20:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/09 07:20:28 INFO mlflow.store.db.utils: Updating database tables
2026/10/09 07:20:37 INFO mlflow.tracking.fluent: Experiment with name 'glof_risk' does not exist. Creating a new experiment.


random_forest {'cv_roc_auc': np.float64(0.906), 'cv_pr_auc': np.float64(0.732), 'cv_recall': 0.69, 'cv_precision': 0.667}
xgboost {'cv_roc_auc': np.float64(0.887), 'cv_pr_auc': np.float64(0.696), 'cv_recall': 0.697, 'cv_precision': 0.593}
AutoML baseline best: lgbm | CV ROC-AUC 0.923
Ablation (Tista-only negatives) CV ROC-AUC: 0.746 - compare with region-matched above; a big gap = geographic shortcut


## 5. Final model, South Lhonak hold-out test, feature importance

In [ ]:
best_name = max([k for k in results if k != 'automl_flaml'], key=lambda k: results[k]['cv_roc_auc'])
final = make_models()[best_name].fit(X, y)
print('Best model:', best_name, results[best_name])

bundle = {'model': final, 'features': FEATURES, 'model_name': best_name, 'metrics': results[best_name],
          'trained_on': {'n_pos': int((y == 1).sum()), 'n_neg': int((y == 0).sum())}}
joblib.dump(bundle, f'{MODELS}/glof_risk_model.joblib')
if HAVE_MLFLOW:
    with mlflow.start_run(run_name=f'final_{best_name}'):
        mlflow.log_params({'model': best_name, 'features': ','.join(FEATURES)})
        mlflow.log_metrics(results[best_name])
        mlflow.log_artifact(f'{MODELS}/glof_risk_model.joblib')

# feature importance
est = final.steps[-1][1] if hasattr(final, 'steps') else final
if hasattr(est, 'feature_importances_'):
    imp = pd.Series(est.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print(imp.round(3).head(10))

# South Lhonak (never seen in training)
lhonak_result = None
if test_ids:
    lid = sorted(test_ids)[0]
    row = table[table['lake_id'] == lid]
    p_l = float(final.predict_proba(row[FEATURES])[:, 1][0])
    tista_ids = set(tista_all['lake_id'])
    all_scores = pd.Series(final.predict_proba(table[table['lake_id'].isin(tista_ids)][FEATURES])[:, 1])
    pct = float((all_scores < p_l).mean() * 100)
    lhonak_result = {'lake_id': lid, 'model_prob': round(p_l, 3), 'percentile_in_tista_basin': round(pct, 1),
                     'area_km2_in_2005_inventory': float(row['area_km2'].iloc[0])}
    print('SOUTH LHONAK (held-out):', lhonak_result)
    print('Note: the inventory is from ~2005, when the lake was much smaller. Part 2 lake-growth is what captures its later expansion.')
else:
    print('South Lhonak was not matched to an inventory lake - check the 3 km match.')

Best model: random_forest {'cv_roc_auc': np.float64(0.90570653970773), 'cv_pr_auc': np.float64(0.7321172805456659), 'cv_recall': 0.6901408450704225, 'cv_precision': 0.6666666666666666}
dist_glacier_km         0.262
glacier_area_5km_km2    0.206
glacier_area_km2        0.104
type_e_o_               0.084
log_area_km2            0.080
type_m_e_               0.077
altitude                0.065
slope_mean              0.049
slope_max               0.046
type_m_o_               0.014
dtype: float64
SOUTH LHONAK (held-out): {'lake_id': 'HKH-08027', 'model_prob': 0.928, 'percentile_in_tista_basin': 98.3, 'area_km2_in_2005_inventory': 0.977}
Note: the inventory is from ~2005, when the lake was much smaller. Part 2 lake-growth is what captures its later expansion.


## 6. Score every Tista lake (+ blend in Part 2 lake growth if available)
Lakes that were in the training set get their out-of-fold prediction (no self-scoring).
**Part 2 hook:** put a CSV at `processed/lake_growth.csv` with columns `lat, lon, growth_pct`
(one row per lake whose area you compared across years, e.g. 14.2 for +14.2%).
If it is missing the growth column is empty and the score is just the model probability.

In [ ]:
tab_t = table[table['lake_id'].isin(set(tista_all['lake_id']))].copy()
tab_t['model_prob'] = final.predict_proba(tab_t[FEATURES])[:, 1]
oof_map = dict(zip(train['lake_id'], oof[best_name]))
tab_t['model_prob'] = [oof_map.get(i, p) for i, p in zip(tab_t['lake_id'], tab_t['model_prob'])]

tab_t['lake_growth_pct'] = np.nan
growth_path = f'{PROC}/lake_growth.csv'
if os.path.exists(growth_path):
    g = pd.read_csv(growth_path)
    gp = gpd.GeoDataFrame(g, geometry=gpd.points_from_xy(g['lon'], g['lat']), crs=4326).to_crs(METRIC_CRS)
    tp = gpd.GeoDataFrame(tab_t[['lake_id']], geometry=gpd.points_from_xy(tab_t['Longitude'], tab_t['Latitude']), crs=4326).to_crs(METRIC_CRS)
    gj = gpd.sjoin_nearest(tp, gp[['growth_pct', 'geometry']], how='left', max_distance=1000).drop_duplicates('lake_id')
    tab_t['lake_growth_pct'] = tab_t['lake_id'].map(gj.set_index('lake_id')['growth_pct'])
    print('Lake growth matched for', int(tab_t['lake_growth_pct'].notna().sum()), 'lakes')
else:
    print('No processed/lake_growth.csv found - growth left empty (see Part 2 hook above).')

# Documented heuristic (NOT learned): 85% model probability + 15% growth signal (caps at +50% growth).
growth_signal = (tab_t['lake_growth_pct'].clip(lower=0, upper=50) / 50).fillna(0)
has_g = tab_t['lake_growth_pct'].notna()
tab_t['risk_score'] = np.where(has_g, 0.85 * tab_t['model_prob'] + 0.15 * growth_signal, tab_t['model_prob']).round(4)
tab_t['risk_level'] = pd.cut(tab_t['risk_score'], [-0.01, 0.35, 0.60, 1.01], labels=['Low', 'Medium', 'High']).astype(str)

No processed/lake_growth.csv found - growth left empty (see Part 2 hook above).


## 7. Isolation Forest — abnormal lakes

In [ ]:
from sklearn.ensemble import IsolationForest
iso_cols = [c for c in ['log_area_km2', 'altitude', 'dist_glacier_km', 'glacier_area_5km_km2', 'slope_mean', 'slope_max'] if c in tab_t.columns]
iso_X = SimpleImputer(strategy='median').fit_transform(tab_t[iso_cols])
iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=RANDOM_STATE).fit(iso_X)
raw_s = -iso.score_samples(iso_X)
tab_t['anomaly_score'] = ((raw_s - raw_s.min()) / (raw_s.max() - raw_s.min() + 1e-9)).round(3)
tab_t['is_anomaly'] = (iso.predict(iso_X) == -1)
print('Anomalous lakes:', int(tab_t['is_anomaly'].sum()), 'of', len(tab_t))

Anomalous lakes: 20 of 401


## 8. Time-series forecasting (rainfall, temperature, lake area)

In [ ]:
import requests, time
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def fetch_power(lat, lon, start='20150101', end='20251231'):
    url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
    params = {'parameters': 'PRECTOTCORR,T2M', 'community': 'AG', 'longitude': lon, 'latitude': lat,
              'start': start, 'end': end, 'format': 'JSON'}
    for _ in range(3):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code == 200:
                df = pd.DataFrame(r.json()['properties']['parameter'])
                df.index = pd.to_datetime(df.index, format='%Y%m%d')
                return df.replace(-999, np.nan)
        except Exception as e:
            print('NASA POWER retry:', e)
        time.sleep(10)
    return None

forecast_rows = []
wx = fetch_power(27.9, 88.18)
if wx is not None:
    monthly = pd.DataFrame({'temperature_c': wx['T2M'].resample('MS').mean(),
                            'rainfall_mm': wx['PRECTOTCORR'].resample('MS').sum()}).dropna()
    for var in ['temperature_c', 'rainfall_mm']:
        s = monthly[var]
        hw = ExponentialSmoothing(s, trend=None, seasonal='add', seasonal_periods=12).fit()
        fc = hw.forecast(12)
        forecast_rows += [{'date': d.strftime('%Y-%m-%d'), 'variable': var, 'value': round(float(v), 2), 'kind': 'history'}
                          for d, v in s.tail(36).items()]
        forecast_rows += [{'date': d.strftime('%Y-%m-%d'), 'variable': var, 'value': round(float(v), 2), 'kind': 'forecast'}
                          for d, v in fc.items()]
    print('Climate forecast built.')
else:
    print('NASA POWER unreachable - climate forecast skipped (re-run this cell later).')

# Lake-area forecast: optional Part 2 hook -> processed/lake_area_timeseries.csv with columns year, area_km2
ts_path = f'{PROC}/lake_area_timeseries.csv'
if os.path.exists(ts_path):
    ts = pd.read_csv(ts_path).sort_values('year')
    coef = np.polyfit(ts['year'], ts['area_km2'], 1)
    last = int(ts['year'].max())
    forecast_rows += [{'date': f'{int(r.year)}-01-01', 'variable': 'lake_area_km2', 'value': round(float(r.area_km2), 4), 'kind': 'history'}
                      for r in ts.itertuples()]
    forecast_rows += [{'date': f'{yr}-01-01', 'variable': 'lake_area_km2', 'value': round(float(np.polyval(coef, yr)), 4), 'kind': 'forecast'}
                      for yr in range(last + 1, last + 4)]
    print('Lake-area forecast built (linear trend, 3 years).')
else:
    print('No processed/lake_area_timeseries.csv - lake-area forecast skipped (Part 2 hook).')

Climate forecast built.
No processed/lake_area_timeseries.csv - lake-area forecast skipped (Part 2 hook).


## 9. Export everything Part 4 needs

In [ ]:
from datetime import datetime, timezone

def clean_name(r):
    return 'South Lhonak Lake' if (lhonak_result and r['lake_id'] == lhonak_result['lake_id']) else f"Lake {r['lake_id']}"

out = pd.DataFrame({
    'lake_id': tab_t['lake_id'], 'name': tab_t.apply(clean_name, axis=1), 'region': 'tista',
    'lat': tab_t['Latitude'], 'lon': tab_t['Longitude'], 'basin': tab_t['Basin'], 'sub_basin': tab_t['Sub_basin'],
    'area_km2': tab_t['area_km2'].round(4), 'altitude_m': tab_t['Altitude'], 'lake_type': tab_t['Type'],
    'dist_to_glacier_km': tab_t['dist_glacier_km'].round(2),
    'model_prob': tab_t['model_prob'].round(4), 'lake_growth_pct': tab_t['lake_growth_pct'],
    'risk_score': tab_t['risk_score'], 'risk_level': tab_t['risk_level'],
    'anomaly_score': tab_t['anomaly_score'], 'is_anomaly': tab_t['is_anomaly'],
    'near_known_burst': tab_t['near_burst'],
    'is_case_study': tab_t['lake_id'].isin(test_ids),
}).sort_values('risk_score', ascending=False)
out.to_csv(f'{EXPORT}/lake_risk.csv', index=False)

# villages with coordinates (joined from Part 1's osm_valley.json by name)
try:
    with open(f'{RAW}/osm_valley.json') as f:
        valley = json.load(f)
    coords = pd.DataFrame([{'name': el.get('tags', {}).get('name:en') or el.get('tags', {}).get('name', 'unnamed'),
                            'lat': el['lat'], 'lon': el['lon']} for el in valley['elements']]).drop_duplicates('name')
    vout = villages.merge(coords, on='name', how='left')
except Exception as e:
    print('Village coordinates not joined:', e)
    vout = villages.assign(lat=np.nan, lon=np.nan)
vout.to_csv(f'{EXPORT}/villages.csv', index=False)

pd.DataFrame(forecast_rows, columns=['date', 'variable', 'value', 'kind']).to_csv(f'{EXPORT}/forecast.csv', index=False)

summary = {'generated_at': datetime.now(timezone.utc).isoformat(), 'demo': False,
           'model': {'name': best_name, 'cv': {k: round(v, 3) for k, v in results[best_name].items()},
                     'all_models': {k: {m_: round(v_, 3) for m_, v_ in v.items()} for k, v in results.items()},
                     'features': FEATURES, **bundle['trained_on']},
           'case_study': lhonak_result,
           'notes': ['Risk scores are relative model outputs (training uses sampled negatives), not calibrated flood probabilities.',
                     'Lake attributes come from the ~2005 ICIMOD inventory; lake growth comes from Part 2 when provided.']}
with open(f'{EXPORT}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('Exported to', EXPORT)
print(os.listdir(EXPORT), '| model in', MODELS)
print(out['risk_level'].value_counts())

Exported to /content/drive/MyDrive/HimalayanGuardian/processed/export
['lake_risk.csv', 'villages.csv', 'forecast.csv', 'summary.json'] | model in /content/drive/MyDrive/HimalayanGuardian/processed/models
risk_level
Low       218
Medium    107
High       76
Name: count, dtype: int64
